# L5c: Introduction to Linear Programming

Linear programming gives us a common language for resource allocation, production planning, and minimum-cost network flow.

> **Learning objectives**
>
> - Identify decision variables, a linear objective, constraints, and bounds.
> - Formulate primal resource-allocation and minimum-cost-flow models.
> - Interpret dual variables as marginal resource values.
> - Separate model formulation, solver execution, and independent validation.


## Setup

The lecture and companion example use the course's single pinned Julia environment.


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## Examples
Today, we will be using the following example(s) to illustrate key concepts:

> [▶ Two Goods Resource Allocation Problem: Apples versus Oranges](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). In this example, we will explore a simple resource allocation problem involving the purchase (and consumption) of two goods: apples and oranges. We will formulate the primal linear programming problem to maximize the utility (satisfaction) while adhering to resource constraints (budget constraint). 

___


## Primal Linear Programming Problems
Suppose you have a _linear_ objective function $O:\mathbb{R}^{n}\to\mathbb{R}$ of the continuous decision variable vector $\mathbf{x}\in\mathbb{R}^{n}$ whose values are constrained by a system of $m$ linear equations. To calculate an optimal value of the decision variable vector $\mathbf{x}$, we can formulate the problem as a _primal linear programming_ problem:
$$
\begin{align*}
\text{minimize/maximize} &\, \sum_{i=1}^{n} c_{i}\;{x}_{i}\\
\text{subject to}~\mathbf{A}\;\mathbf{x} &\leq \mathbf{b}\quad\mathbf{A}\in\mathbb{R}^{m\times{n}}\,\text{and}\,\mathbf{b}\in\mathbb{R}^{m}\\
~x_{i}&\geq {0}\qquad{i=1,2,\dots,n}
\end{align*}
$$
where $c_{i}\in\mathbb{R}$ are the coefficients of the objective function, $x_{i}\in\mathbb{R}$ are the decision variables, and $\mathbf{A}$ and $\mathbf{b}$ are the constraint matrix and right-hand side vector, respectively. The goal is to minimize or maximize the objective function while satisfying the constraints.

Let's look at a few simple example problems to illustrate the primal linear programming formulation.

### Consumer Choice Problems as Linear Programs
Suppose you are a consumer with a set of products you can purchase, and you want to maximize your utility (satisfaction) from these products while staying within your budget. This is a classic example of a resource allocation task that can be formulated as a primal linear programming problem.

> __Formulation__: Let there be $n$ products available for purchase. For each product $i$, let $u_{i}$ be the utility score and $c_{i}$ be the cost per unit. The consumer has a budget $I$ that they can allocate among these products. Let $x_{i}$ be the number of units of product $i$ that the consumer purchases; the total cost of the products they purchase is given by the expression $\sum_{i=1}^{n} c_{i}\;{x}_{i}$.
> Finally, the consumer's utility function is defined as a linear combination of the utility scores of the products they purchase: $U\left(x_{1},\dots,x_{n}\right) = \sum_{i=1}^{n} u_{i}\;{x}_{i}$.

Putting all this together, we can formulate the consumer choice problem as the _primal_ linear program:
$$
\begin{align*}
\text{maximize} &\, \sum_{i=1}^{n} u_{i}\;{x}_{i} \\
\text{subject to}~\sum_{i=1}^{n} c_{i}\;{x}_{i}& \leq I\\
~x_{i}&\geq{0}\qquad{i=1,2,\dots,n}
\end{align*}
$$
The optimal solution to this problem (if it exists) will give the consumer the optimal number of units of each product to purchase in order to maximize their utility while staying within their budget. In a similar way, we can formulate other resource allocation problems such as production planning, transportation, and network flow as primal linear programming problems.

> __Example__
> 
> [▶ Two Goods Resource Allocation Problem: Apples versus Oranges](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). In this example, we will explore a simple resource allocation problem involving the purchase (and consumption) of two goods: apples and oranges. We will formulate the primal linear programming problem to maximize the utility (satisfaction) while adhering to resource constraints (budget constraint). 

___

### Minimum Cost Network Flow Problems as Linear Programs
Another classic example of a resource allocation problem that can be formulated as a primal linear programming problem is the minimum cost maximum flow problem. In this problem, we have a directed (bipartite) graph with nodes representing sources, sinks, and intermediate nodes representing a matching process. The edges represent the flow of goods or resources between these nodes. Each edge has a capacity (the maximum amount of flow that can pass through it) and a cost per unit of flow.

> __Formulation__: Let the directed graph be represented as $G = (\mathcal{V}, \mathcal{E})$, where $\mathcal{V}$ is the set of vertices (nodes) and $\mathcal{E}$ is the set of edges. Each edge $j \in \mathcal{E}$ has a capacity $c_j$ and a cost (weight) $w_j$ per unit of flow. Let $f_j$ be the flow on edge $j$, and let $s$ be the source node and $t$ be the sink node. The goal is to maximize the flow from the source to the sink while minimizing the total cost of the flow.

We use an incidence matrix formulation where $\mathbf{A} \in \mathbb{R}^{|\mathcal{V}| \times |\mathcal{E}|}$ represents the graph structure. For node $i$ and edge $j$:
- $A_{ij} = 1$ if edge $j$ is incoming to node $i$
- $A_{ij} = -1$ if edge $j$ is outgoing from node $i$  
- $A_{ij} = 0$ otherwise

Putting all this together, we can formulate the minimum cost maximum flow problem as the _primal_ linear program:
$$
\begin{align*}
\text{minimize} &\, \sum_{j \in \mathcal{E}} w_j f_j \\
\text{subject to} \quad \mathbf{A}\mathbf{f} &= \mathbf{b}\\
~0 \leq f_j &\leq c_j \quad\forall j \in \mathcal{E}
\end{align*}
$$
where $\mathbf{f} \in \mathbb{R}^{|\mathcal{E}|}$ is the vector of flows on each edge, and $\mathbf{b} \in \mathbb{R}^{|\mathcal{V}|}$ is the right-hand side vector with:
$$
b_i = \begin{cases}
-F & \text{if } i = s \text{ (source generates flow)} \\
F & \text{if } i = t \text{ (sink consumes flow)} \\
0 & \text{otherwise (flow conservation)}
\end{cases}
$$
where $F$ is the total flow from the source to the sink. The optimal solution to this problem (if it exists) will give the flow on each edge that minimizes the total cost while satisfying the flow conservation constraints and capacity constraints.
___


## Dual Linear Programming Problems
Having defined primal linear programs, we now turn to their duals, alternative formulations that offer a different viewpoint on the same optimization. You can think of it as viewing the primal through a different lens.

If the _primal problem_ has the form:
$$
\begin{align*}
\text{maximize} &\, \sum_{i=1}^{n} c_{i}\;{x}_{i}\\
\text{subject to}~\sum_{i=1}^{n} A_{i,j}\;{x}_{i} &\leq b_{j}\quad j=1,2,\dots,m\\
~x_{i}&\geq {0}\qquad{i=1,2,\dots,n}
\end{align*}
$$
then the _dual problem_ has the form:
$$
\begin{aligned}
\text{minimize}\quad & \sum_{j=1}^{m} b_{j}\,y_{j}\\
\text{subject to}\quad & \sum_{j=1}^{m} A_{i,j}\,y_{j}\;\ge\;c_{i}
\quad&&i=1,2,\dots,n,\\
&y_{j}\;\ge\;0
\quad&&j=1,2,\dots,m.
\end{aligned}
$$

### What has changed?
There are several key differences between the primal and dual linear programming problems:
1. The objective function flips (maximum ⇒ minimum or minimum ⇒ maximum).
2. Primal objective coefficients $c_i$ become the dual right-hand side constants.
3. Primal right-hand side constants $b_j$ become the dual objective coefficients.
4. The $m\times n$ constraint matrix $A$ is transposed in the dual (so $A^\top$ appears).
5. The number of variables and constraints swap: the primal has $n$ variables, $m$ constraints, and the dual has $m$ variables and $n$ constraints.
6. Each primal constraint $a_j^\top x \le b_j$ gives a dual variable $y_j$. Each primal variable $x_i$ gives a dual constraint $(A^\top y)_i \ge c_i$.
7. Inequality directions and sign restrictions invert for the constraints: A $\le$ constraint in the primal gives rise to a $\ge$ constraint in the dual (and vice versa).
8. Equality constraints in the primal become free variables in the dual, i.e., $a_j^T x = b_j$ gives rise to a dual variable $y_j$ that is free (no sign restriction), while a dual constraint $A^\top y \ge c$ gives rise to a primal variable $x_i$ that is free.

Finally, the solutions of the primal and dual problems are related by the concept of __duality__. For a primal problem: $\max\{\,c^T x : A x \le b,\;x\ge0\}$ and its corresponding dual problem: $\min\{\,b^T y : A^T y \ge c,\;y\ge0\}$, the solutions are related:
* __Weak duality__: For any primal feasible $x$ and dual feasible $y$, we have $c^T x \le b^T y$. Thus, the primal optimum is always bounded above by the dual optimum. The difference between the two is called the _duality gap_.
* __Strong duality__: If both primal and dual are feasible and have finite optimal values, then $\max\{\,c^T x \} = \min\{\,b^T y\}$, i.e., the _duality gap is zero_. This means that the optimal values of the primal and dual problems are equal.

___


## How an LP solver fits the modeling workflow

The modeler supplies decision variables, a linear objective, linear constraints, and variable bounds. A solver then returns a status and a candidate solution. Our responsibility is to check the status, recompute the objective, and verify feasibility before interpreting the answer.

The supporting revised-simplex notebook opens one solver-internals window: basic variables select a corner, reduced costs identify a potentially improving direction, and a ratio test preserves feasibility. Interior-point derivations and duality proofs are deeper-dive material rather than required Week 5 content.


## Summary

- An LP separates decisions, objective coefficients, constraints, and bounds.
- The primal describes activities; the dual assigns marginal values to limiting resources.
- Network-flow conservation is an incidence-matrix equality, so minimum-cost flow is an LP.
- Solver status and an independent feasibility check are part of the result.
